
# 01 事件驱动回测与撮合

> **本章要解决的问题**：向量化回测只回答“如果按收盘价调仓，收益是多少”；真实交易里，订单可能没成交、只成交一半、被拒单、被追保。这些差异有多大？一个订单从发出到进账本，中间到底发生了什么？

## 本章导读

前面的章节大多用向量化回测：把信号和目标权重算出来，乘上收益，得到净值曲线。它快、够用，但它默认“想成交就能按某个价格成交”。一旦策略依赖盘口、限价、部分成交，或者需要和原生回放对账，向量化就不够了，需要**事件驱动**的回测器：

1. 事件循环：行情 → 信号 → 订单 → 撮合 → 成交 → 账本；
2. 订单语义：限价/市价、撤单、部分成交、拒单；
3. 账本：逐笔记录成交、持仓、现金、权益，期货还要算保证金；
4. 成本：手续费、滑点、冲击，三者口径不同；
5. 对账：回测输出与 WT-plus 原生回放按“版本 + 交易日 + 成本场景”绑定核对。

一句话：**向量化回答“策略赚多少”，事件驱动回答“这些钱是怎么一笔一笔赚到的”**。

### 学习目标

- 给定一份订单和一张盘口快照，能按价格优先、时间优先写出撮合过程，并说清限价单为什么不成交。
- 给定一串成交，能逐笔维护现金、持仓与权益，并算出手续费、滑点与期货保证金占用。
- 给定一个交易信号，能把它翻译成事件队列中的订单事件，并用账本回放出一笔完整交易。
- 给定手续费、滑点与冲击参数，能评估不同成本假设对净收益的影响，并解释平方根冲击的量级。
- 给定回测输出与原生回放收据，能按“版本 + 交易日 + 成本场景”对账，定位笔数、数量、费用差异。
- 给定一次对账失败，能判断问题出在撮合价、成本口径还是成交缺失。

### 先修

第 14 章的 WT 原生回放与对账；第 31 章的订单容量；第 38 章的运行版本与 manifest；会 Python 基础（类、队列、字典）与 pandas 基本操作。

## 在量化研究中的位置

- **谁用**：策略研究员、交易系统开发、对账与风控、平台运维。
- **输入**：K 线/盘口快照、交易信号、订单参数、成本模型、原生回放收据。
- **输出**：逐笔成交与账本、净值曲线、成本分解、与回放的对账结论。
- **在流程的哪一步**：因子与组合之后、实盘之前；把“信号”翻译成“订单与成交”的最后一层。
- **失败模式**：假设无限流动性、忽略部分成交与拒单、成本口径混用、用不同版本的数据对账。
- **最小工作流**：定事件顺序 → 写撮合规则 → 维护账本 → 加成本 → 跑回测 → 与回放对账。

### 环境准备



In [1]:

import heapq
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("pandas", pd.__version__, "| numpy", np.__version__)


pandas 3.0.2 | numpy 2.4.4



## 1. 事件驱动 vs 向量化：回测在模拟什么

向量化回测的隐含假设是“**价格是外生的，成交是必然的**”：我算出目标权重，就能在那一刻按某个价格全部换过去。事件驱动回测反过来，把成交当成一个需要被**撮合出来**的结果：

1. **行情事件**：新的一根 K 线或一条盘口快照到达；
2. **信号事件**：策略根据行情算出目标持仓；
3. **订单事件**：把目标持仓的差额翻译成订单，写清方向、数量、限价或市价；
4. **撮合事件**：撮合器根据盘口判断能成交多少、成交在什么价；
5. **账本事件**：成交进入现金与持仓账本，更新权益。

事件顺序本身就是模型假设。比如“信号在 t 收盘产生、t+1 开盘成交”与“t 收盘前成交”差一天，在高频策略里可能差出全部 alpha。本章统一约定：**信号在 t 收盘产生，订单在 t+1 根 K 线成交**，并在报告里写清这一点。

和向量化相比，事件驱动慢，但能回答三个向量化回答不了的问题：**这笔订单能不能成交、成交在什么价、成交之后账上还剩多少钱**。

---

## 2. 订单与撮合：限价、市价、撤单、部分成交、拒单

### 2.1 订单的字段

一张订单最少要写清：方向（买/卖）、数量、价格（市价单为空）、类型（限价/市价）、有效期（当日/撤销前有效）、状态。状态机通常长这样：

| 状态 | 含义 | 可以从哪来 |
|---|---|---|
| new | 已报，未成交 | 初始 |
| partial | 部分成交 | new / partial |
| filled | 全部成交 | new / partial |
| cancelled | 已撤单 | new / partial |
| rejected | 被拒绝 | 初始 |

### 2.2 撮合规则

本章用一个简化盘口：买方出价从高到低、卖方要价从低到高排列，每个价位带数量。撮合规则是：

1. 市价单：从对手方最优价位开始逐档吃，直到数量满足；
2. 限价买：只吃“卖价 ≤ 限价”的档位，吃完仍有剩余就挂着；
3. 限价卖：只吃“买价 ≥ 限价”的档位；
4. 数量为 0 或非法：直接拒单；
5. 撤单：把未成交部分标记为 cancelled，不产生成交。

注意几个真实世界的细节：盘口是**快照**，撮合时用的是快照里那一瞬间的深度；部分成交后剩余的订单仍在队列里，下一帧行情还会继续被撮合。本章把“挂着继续撮合”简化为“本次不再撮合”，但状态保留为 partial / new。

### 2.3 代码：撮合一个订单



In [2]:

@dataclass
class Order:
    order_id: int
    side: str
    qty: int
    price: float
    kind: str = "limit"
    status: str = "new"
    filled: int = 0


@dataclass
class Fill:
    order_id: int
    side: str
    qty: int
    price: float


def match_order(order, bids, asks):
    if order.qty <= 0:
        order.status = "rejected"
        return [], 0
    book = asks if order.side == "buy" else bids
    fills, remaining = [], order.qty
    for level_price, level_qty in book:
        if remaining <= 0:
            break
        if order.kind == "limit":
            if order.side == "buy" and level_price > order.price:
                break
            if order.side == "sell" and level_price < order.price:
                break
        qty = min(remaining, level_qty)
        fills.append(Fill(order.order_id, order.side, qty, level_price))
        remaining -= qty
    order.filled += sum(f.qty for f in fills)
    if remaining == 0:
        order.status = "filled"
    elif order.filled > 0:
        order.status = "partial"
    return fills, remaining


bids = [(100.05, 80), (100.00, 200), (99.95, 300)]
asks = [(100.10, 100), (100.15, 150), (100.20, 300)]

o1 = Order(1, "buy", 150, 0.0, kind="market")
f1, r1 = match_order(o1, bids, asks)
print("市价买 150：", [(f.qty, f.price) for f in f1], "| 剩余", r1, "| 状态", o1.status)

o2 = Order(2, "buy", 100, 100.05)
f2, r2 = match_order(o2, bids, asks)
print("限价买 100@100.05：成交", [(f.qty, f.price) for f in f2], "| 状态", o2.status)
o2.status = "cancelled"
print("撤单后状态：", o2.status)

o3 = Order(3, "sell", 120, 99.95)
f3, r3 = match_order(o3, bids, asks)
print("限价卖 120@99.95：", [(f.qty, f.price) for f in f3], "| 状态", o3.status)

o4 = Order(4, "buy", 0, 99.0)
f4, r4 = match_order(o4, bids, asks)
print("零数量订单：成交", len(f4), "| 状态", o4.status)

fills_df = pd.DataFrame([vars(f) for f in f1 + f3])
print(fills_df.to_string(index=False))


市价买 150： [(100, 100.1), (50, 100.15)] | 剩余 0 | 状态 filled
限价买 100@100.05：成交 [] | 状态 new
撤单后状态： cancelled
限价卖 120@99.95： [(80, 100.05), (40, 100.0)] | 状态 filled
零数量订单：成交 0 | 状态 rejected
 order_id side  qty  price
        1  buy  100 100.10
        1  buy   50 100.15
        3 sell   80 100.05
        3 sell   40 100.00



**读数**：市价买 150 手吃穿两档（100 手 @100.10、50 手 @100.15），成交均价约 100.1167；限价买 100@100.05 因为最优卖价 100.10 高于限价，一股没成交，状态保持 new，撤单后变成 cancelled；限价卖 120@99.95 吃两档买单（80 手 @100.05、40 手 @100.00）全部成交；数量为 0 的订单被直接拒单。

**常见错误**：

1. 限价单按“限价”成交——实际只能按对手方更优的价位成交，限价只是上限/下限。
2. 忽略部分成交——回测里“100 手全成交”，实盘只成了 40 手。
3. 撤单不更新剩余量——后续行情又把它当成新订单重新撮合。

---



## 3. 逐笔账本：成交、持仓、现金、权益、保证金

回测的“账”是三个量的守恒关系：

$$\text{权益} = \text{现金} + \text{持仓} \times \text{最新价}.$$

每发生一笔成交，账本要同时更新五个东西：现金（买入减少、卖出增加）、持仓（买入增加、卖出减少）、成交均价、手续费、权益。买入时现金流出 = 成交额 + 手续费；卖出时现金流入 = 成交额 − 手续费。账本要能回答“这笔交易到底赚了多少”，而不只是“期末净值多少”。

### 3.1 滑点：先改价格，再算成交

滑点建模的是“你的成交价比参考价差多少”。常见写法是按 bps：

$$P_{\text{成交}} = P_{\text{参考}} \times \left(1 \pm \frac{s}{10^4}\right),$$

买入取加号、卖出取减号。它可以是常数、按波动率放大，也可以按订单量放大。注意：滑点只影响价格，不改变数量；手续费只影响现金，不改变价格。

### 3.2 保证金：期货账本的额外约束

股票是全额付款，期货是保证金交易。一笔期货持仓的占用资金与风险不是同一个数：

$$\text{名义价值} = \text{价格} \times \text{乘数} \times \text{手数},\qquad \text{保证金} = \text{名义价值} \times \text{保证金率}.$$

保证金率通常 8%–15%，交易所会随波动调整。日内价格越不利，浮动亏损先从“可用资金”里扣，扣到不足就追保甚至强平——这是向量化回测通常不模拟、但实盘一定会遇到的一层。

### 3.3 代码：一笔完整交易的账本



In [3]:

class Ledger:
    def __init__(self, cash=1_000_000.0, fee_bps=5.0, slip_bps=2.0):
        self.cash = cash
        self.position = 0
        self.fee_bps = fee_bps
        self.slip_bps = slip_bps
        self.rows = []

    def execute(self, side, qty, ref_price, last_price):
        sign = 1 if side == "buy" else -1
        px = ref_price * (1 + sign * self.slip_bps / 1e4)
        notional = px * qty
        fee = notional * self.fee_bps / 1e4
        self.cash -= sign * notional
        self.cash -= fee
        self.position += sign * qty
        self.rows.append({
            "side": side, "qty": qty, "price": round(px, 4), "notional": round(notional, 2),
            "fee": round(fee, 3), "cash": round(self.cash, 2), "position": self.position,
            "equity": round(self.cash + self.position * last_price, 2),
        })
        return px, fee

    def equity(self, price):
        return self.cash + self.position * price


led = Ledger(cash=1_000_000.0, fee_bps=5.0, slip_bps=2.0)
print("买入 100 @100.00：成交价与手续费", led.execute("buy", 100, 100.0, 100.0))
print("卖出 100 @102.00：成交价与手续费", led.execute("sell", 100, 102.0, 102.0))
print(pd.DataFrame(led.rows).to_string(index=False))
print("期末权益", round(led.equity(102.0), 2), "| 净盈亏", round(led.equity(102.0) - 1_000_000.0, 2))
print("总手续费", round(sum(r["fee"] for r in led.rows), 3))
slip_cost = abs(led.rows[0]["price"] - 100.0) * 100 + abs(led.rows[1]["price"] - 102.0) * 100
print("总滑点成本约", round(slip_cost, 2))

multiplier = 10
fut_price = 3500.0
lots = 10
margin_rate = 0.10
margin = fut_price * multiplier * lots * margin_rate
print("螺纹钢 10 手：名义价值", fut_price * multiplier * lots, "| 保证金占用", margin,
      "| 手续费约", round(fut_price * multiplier * lots * 1e-4, 2))


买入 100 @100.00：成交价与手续费 (100.02, 5.001)
卖出 100 @102.00：成交价与手续费 (101.9796, 5.09898)
side  qty    price  notional   fee       cash  position     equity
 buy  100 100.0200  10002.00 5.001  989993.00       100  999993.00
sell  100 101.9796  10197.96 5.099 1000185.86         0 1000185.86
期末权益 1000185.86 | 净盈亏 185.86
总手续费 10.1
总滑点成本约 4.04
螺纹钢 10 手：名义价值 350000.0 | 保证金占用 35000.0 | 手续费约 35.0



**读数**：买入 100 股按 100.02 成交（滑点 2bp）、手续费 5.001；卖出 100 股按 101.9796 成交、手续费 5.099。毛利 $(102-100)\times100=200$ 元，减去手续费 10.099 与滑点成本 4.04，净盈亏 185.86 元，三笔账对得上。螺纹钢 10 手名义价值 35 万元，10% 保证金率只占用 3.5 万元——**同样的“仓位”，期货的资金占用和风险敞口不是一回事**。

**常见错误**：

1. 滑点方向和买卖反了——买入应该成交更贵，卖出应该更便宜。
2. 手续费按“手数”而不是“成交额”算，或者反过来，口径混用。
3. 用保证金当亏损上限——保证金只是门槛，亏损可以超过它。

---

## 4. 最小事件驱动回测器

### 4.1 事件顺序

把回测器写成一个循环：每个 bar 依次处理“**上一根产生的订单** → 记录权益 → 根据本根收盘产生新订单”。这个顺序决定了成交价：

1. bar t 收盘：用截至 t 的数据算信号；
2. bar t+1：先按 t+1 的价格撮合订单，再记录权益；
3. 信号里只能用到 $t$ 及之前的数据，否则就是未来函数。

真实系统里还有排队、撤单、盘口变化；本章只保留最小闭环，重点是把“信号 → 订单 → 成交 → 账本”串起来。

### 4.2 代码：双均线策略的事件驱动回放



In [4]:

def run_event_backtest(close, cash=1_000_000.0, fee_bps=5.0, slip_bps=2.0,
                       fast=5, slow=20, impact_eta=0.0, adv=None, sigma_daily=None):
    ledger = Ledger(cash=cash, fee_bps=fee_bps, slip_bps=slip_bps)
    events = [(t, 0, "market") for t in range(len(close))]
    heapq.heapify(events)
    fills, equity_curve = [], []
    pending = None
    fee_total = slip_total = impact_total = 0.0
    while events:
        t, _, kind = heapq.heappop(events)
        if kind != "market":
            continue
        price = close[t]
        if pending is not None:
            side, qty = pending
            extra_bps = 0.0
            if impact_eta > 0 and adv:
                extra_bps = impact_eta * sigma_daily * np.sqrt(qty / adv) * 1e4
                impact_total += extra_bps / 1e4 * abs(qty) * price
            base_slip = ledger.slip_bps
            ledger.slip_bps = slip_bps + extra_bps
            px, fee = ledger.execute(side, qty, price, price)
            ledger.slip_bps = base_slip
            fee_total += fee
            slip_total += abs(px - price) * abs(qty) - extra_bps / 1e4 * abs(qty) * price
            fills.append({"bar": t, "side": side, "qty": qty, "price": round(px, 4), "fee": round(fee, 3)})
            pending = None
        equity_curve.append((t, ledger.equity(price)))
        if t >= slow:
            fast_ma = close[t - fast + 1: t + 1].mean()
            slow_ma = close[t - slow + 1: t + 1].mean()
            target = 5000 if fast_ma > slow_ma else 0
            if target != ledger.position:
                side = "buy" if target > ledger.position else "sell"
                pending = (side, abs(target - ledger.position))
    if ledger.position:
        px, fee = ledger.execute("sell", ledger.position, close[-1], close[-1])
        fee_total += fee
        slip_total += abs(px - close[-1]) * ledger.position
        fills.append({"bar": len(close) - 1, "side": "sell", "qty": ledger.position,
                      "price": round(px, 4), "fee": round(fee, 3)})
        equity_curve[-1] = (len(close) - 1, ledger.equity(close[-1]))
    eq = pd.Series([e for _, e in equity_curve])
    dd = (eq / eq.cummax() - 1).min()
    return {
        "fills": pd.DataFrame(fills), "equity": eq,
        "final": eq.iloc[-1], "ret": eq.iloc[-1] / cash - 1, "mdd": dd,
        "fee_total": fee_total, "slip_total": slip_total, "impact_total": impact_total,
    }


rng = np.random.default_rng(24101)
n_bars = 240
drift = np.r_[np.full(120, 0.0016), np.full(120, -0.0009)]
close = 100.0 * np.exp(np.cumsum(drift + rng.normal(0.0, 0.008, n_bars)))
res = run_event_backtest(close)
print("成交笔数", len(res["fills"]), "| 期末权益", round(res["final"], 2),
      "| 累计收益", f"{res['ret']:.2%}", "| 最大回撤", f"{res['mdd']:.2%}")
print("总手续费", round(res["fee_total"], 2), "| 总滑点成本", round(res["slip_total"], 2))
print(res["fills"].head(4).to_string(index=False))


成交笔数 14 | 期末权益 995007.49 | 累计收益 -0.50% | 最大回撤 -6.71%
总手续费 3959.28 | 总滑点成本 1583.71
 bar side  qty    price     fee
  21  buy 5000 108.6670 271.667
  31 sell 5000 105.2986 263.246
  46  buy 5000 107.5657 268.914
  82 sell 5000 112.7764 281.941



**读数**：240 根 K 线里双均线策略成交 14 笔，期末权益 995,007.49，累计收益 −0.50%，最大回撤 −6.71%；手续费 3,959.28、滑点成本 1,583.71。注意这是“信号在 t 收盘产生、t+1 成交”的结果：如果偷懒用同一根 K 线的收盘价成交，收益会被系统性高估，这类错误在事件驱动里最好被抓出来。

**常见错误**：

1. 用产生信号的那根 K 线价格成交——未来函数。
2. 事件顺序不固定（先记账还是先成交），同一份数据跑出不同结果。
3. 期末不强制平仓，账上还留着浮盈浮亏就报“已实现收益”。

---



## 5. 费用与冲击：显性成本与隐性成本

回测里的成本通常分两层：

- **显性成本**：手续费、过户费、印花税——按成交额或手数收取，数据确定、容易算；
- **隐性成本**：滑点与市场冲击——你的成交把价格推差了多少，没有现成数据，只能建模。

### 5.1 手续费与滑点

手续费按成交额比例（bps）或按手数收；股票卖出还有印花税。滑点通常按 bps 建模，或者按“波动率的一个比例”。两者都对收益有线性拖累：成交额越大、笔数越多，拖累越大。

### 5.2 平方根冲击

当订单量占日均成交量（ADV）的比例上升时，冲击成本近似按**平方根**增长：

$$\text{impact} \approx \eta \cdot \sigma_{\text{daily}} \cdot \sqrt{\frac{Q}{V}},$$

其中 $Q$ 是订单量、$V$ 是日均成交量、$\sigma_{\text{daily}}$ 是日波动率、$\eta$ 是经验系数（0.5–1）。换成 bps 就是再乘 $10^4$。它的含义是：订单量翻 4 倍，冲击只翻 2 倍——所以大单不一定成比例更贵，但绝对金额依然线性上升。

### 5.3 代码：成本情景对比



In [5]:

def impact_bps(order_qty, adv, sigma_daily, eta=0.7):
    return eta * sigma_daily * np.sqrt(order_qty / adv) * 1e4


adv = 2_000_000
sigma = 0.02
print("订单量 -> 冲击成本（bps）")
for q in [1_000, 5_000, 20_000, 100_000]:
    print(f"  {q:>7,} 股：{impact_bps(q, adv, sigma):7.2f} bps")

scenarios = [
    ("不计成本", 0.0, 0.0, 0.0),
    ("手续费5bp+滑点2bp", 5.0, 2.0, 0.0),
    ("再加平方根冲击", 5.0, 2.0, 0.7),
]
rows = []
for name, fee_bps, slip_bps, eta in scenarios:
    r = run_event_backtest(close, fee_bps=fee_bps, slip_bps=slip_bps,
                           impact_eta=eta, adv=adv if eta else None,
                           sigma_daily=sigma if eta else None)
    rows.append({"情景": name, "期末权益": round(r["final"], 2), "收益率": f"{r['ret']:.2%}",
                 "手续费": round(r["fee_total"], 2), "滑点": round(r["slip_total"], 2),
                 "冲击": round(r["impact_total"], 2)})
print(pd.DataFrame(rows).to_string(index=False))


订单量 -> 冲击成本（bps）
    1,000 股：   3.13 bps
    5,000 股：   7.00 bps
   20,000 股：  14.00 bps
  100,000 股：  31.30 bps
          情景       期末权益    收益率     手续费      滑点      冲击
        不计成本 1000550.47  0.06%    0.00    0.00    0.00
手续费5bp+滑点2bp  995007.49 -0.50% 3959.28 1583.71    0.00
     再加平方根冲击  989464.50 -1.05% 3959.28 1583.71 5542.99



**读数**：5,000 股的订单在 200 万股 ADV 下冲击约 7 bps；订单放大到 10 万股时冲击约 31.3 bps，接近平方根增长。三种成本假设下：不计成本收益 +0.06%；加了 5 bp 手续费与 2 bp 滑点后收益变成 −0.50%（手续费 3,959.28、滑点 1,583.71）；再加平方根冲击后收益 −1.05%（冲击成本 5,542.99）。**策略的“毛收益”可能被成本直接吃成负的**，而冲击往往是最大的一块。

**常见错误**：

1. 把冲击当成固定滑点，忽略订单规模的影响。
2. 只算手续费，把冲击留给实盘“碰运气”。
3. 用小资金回测的冲击参数去推大资金容量。

---

## 6. 与 WT-plus 原生回放对账

### 6.1 对账的绑定键

回测和原生回放（第 14 章）是两套独立实现，必须能对上账，否则无法判断差异来自策略、撮合还是成本。对账的绑定键至少三个：

1. **版本**：策略/代码版本一致；
2. **交易日**：同一个交易日；
3. **成本场景**：手续费、滑点、冲击参数一致。

只按日期合并是常见的坑：两个版本、两套成本假设的记录被混在一起，差异永远对不清。

### 6.2 对账口径与容差

按日聚合后比较四列：成交笔数、成交数量、成交额、手续费。容差要写死：数量必须精确相等；金额允许浮点误差（如 0.01 元）。差异按类型归档：**撮合价不同**（数量同、金额不同）、**费用口径不同**（数量同、费用不同）、**缺单/多单**（笔数或数量不同）。

### 6.3 代码：一次带差异的对账



In [6]:

def daily_summary(result, version="WT-v2.1", cost_scenario="fee5_slip2"):
    df = result["fills"].copy()
    dates = pd.bdate_range("2023-01-02", periods=n_bars).strftime("%Y-%m-%d")
    df["trade_date"] = dates[df["bar"].to_numpy()]
    df["notional"] = df["qty"] * df["price"]
    g = df.groupby("trade_date").agg(
        trades=("qty", "size"), qty=("qty", "sum"), notional=("notional", "sum"), fee=("fee", "sum")
    ).reset_index()
    g["version"] = version
    g["cost_scenario"] = cost_scenario
    return g


bt = daily_summary(res)
native = bt.copy()
native["fee"] = native["fee"].round(2)
native.loc[native.index[0], "fee"] = round(native.loc[native.index[0], "fee"] + 0.50, 2)
native = native.drop(index=native.index[1]).reset_index(drop=True)
native.loc[len(native) - 1, "qty"] = native.loc[len(native) - 1, "qty"] + 100

key = ["version", "trade_date", "cost_scenario"]
merged = pd.merge(bt, native, on=key, how="outer", suffixes=("_bt", "_native"), indicator=True)
merged["qty_diff"] = merged["qty_native"] - merged["qty_bt"]
merged["fee_diff"] = merged["fee_native"] - merged["fee_bt"]
bad = merged[(merged["_merge"] != "both")
             | (merged["qty_diff"].abs() > 0)
             | (merged["fee_diff"].abs() > 0.01)]
print("回测日数", len(bt), "| 对账日数", len(merged), "| 差异日数", len(bad))
print(bad[["trade_date", "_merge", "qty_bt", "qty_native", "fee_bt", "fee_native"]].to_string(index=False))
print("对账结论：", "通过" if len(bad) == 0 else f"发现 {len(bad)} 个差异日，需人工核对")


回测日数 14 | 对账日数 14 | 差异日数 3
trade_date    _merge  qty_bt  qty_native  fee_bt  fee_native
2023-01-31      both    5000      5000.0 271.667      272.17
2023-02-14 left_only    5000         NaN 263.246         NaN
2023-12-01      both    5000      5100.0 290.069      290.07
对账结论： 发现 3 个差异日，需人工核对



**读数**：回测与原生回放各有 14 个成交日；按“版本 + 交易日 + 成本场景”绑定后，发现 3 个差异日：第一天手续费差 0.50 元（费用口径/舍入不同）、第二天回测有单而回放没有（缺单）、最后一天数量差 100 股（部分成交没对上）。这类输出就是排查的起点：**差异必须落到“哪一天、哪一列、差多少”，而不是只报一个总收益差**。

**常见错误**：

1. 只按交易日对账，不绑版本与成本场景。
2. 用总收益差代替逐日对账，看不出差异发生在哪天。
3. 容差拍脑袋（比如金额允许差 100 元），把真差异放过去。

---

## 本章小结

### 一张表

| 环节 | 要模拟什么 | 本章做法 |
|---|---|---|
| 事件循环 | 行情 → 信号 → 订单 → 成交 → 账本 | 逐 bar 队列，t+1 成交 |
| 撮合 | 限价/市价、部分成交、拒单、撤单 | 简化盘口 + 价格优先 |
| 账本 | 现金、持仓、权益、保证金 | 逐笔记录，权益守恒 |
| 成本 | 手续费、滑点、冲击 | bps + 平方根冲击 |
| 对账 | 与原生回放逐日比对 | 版本 + 交易日 + 成本场景 |

### 八句话

1. 向量化回答“赚多少”，事件驱动回答“怎么赚的”。
2. 限价单不一定成交，成交价也不等于限价。
3. 部分成交与拒单是常态，不是异常。
4. 滑点改价格，手续费改现金，两笔账要分开记。
5. 保证金是门槛，不是最大亏损。
6. 信号在 t 收盘产生、t+1 成交，顺序写进报告。
7. 冲击按平方根增长，大资金必须单独建模。
8. 对账要绑定版本、交易日与成本场景。

---

## 本章习题

**1.** 限价买单挂在 100.05，当前最优卖价 100.10，会成交吗？为什么？

**2.** 市价买 150 手吃两档（100 手 @100.10、50 手 @100.15），成交均价是多少？

**3.** 逐笔账本里，买入 100 股 @100、2bp 滑点、5bp 手续费，现金变化是多少？

**4.** 螺纹钢 10 手、价格 3500、乘数 10、保证金率 10%，保证金占用与名义价值各是多少？

**5.** 为什么事件驱动回测里“信号在 t 产生、t 成交”是未来函数？

**6.** 平方根冲击模型里，订单量翻 4 倍、冲击大约变几倍？

**7.** 回测与原生回放对账，为什么要绑定“版本 + 交易日 + 成本场景”三个键？

**8.** 对账发现某天数量差 100 股、金额差约 1 万元，最可能是什么原因？

<details><summary>参考答案</summary>

**1.** 不会成交。限价买只吃卖价不高于 100.05 的档位，最优卖价 100.10 高于限价，订单保持未成交。

**2.** $(100\times100.10+50\times100.15)/150\approx100.1167$。

**3.** 成交价 $100\times(1+2/10^4)=100.02$，成交额 10,002，手续费 5.001，现金减少 10,007.001。

**4.** 名义价值 $3500\times10\times10=350{,}000$ 元；保证金 $350{,}000\times10\%=35{,}000$ 元。

**5.** 因为 t 时刻的信号用到了 t 的收盘价，而 t 的成交也发生在 t；实盘要等 t 收盘后下单，最早 t+1 成交。

**6.** 约 2 倍（$\sqrt{4}=2$），不是 4 倍；但绝对金额仍随订单量上升。

**7.** 版本不同代码不同、成本场景不同费用口径不同；只按日期合并会把不同口径的记录混在一起，差异无法归因。

**8.** 最可能是部分成交：回测假设全部成交，原生回放只成了部分；也可能是回放多/少了一笔同向订单。需要看订单状态与成交明细再确认。

</details>

---

## 参考资料

- WonderTrader-plus：<https://github.com/newbigdeng/WonderTrader-plus>
- wtpy-plus：<https://github.com/newbigdeng/wtpy-plus>
- Almgren, R. & Chriss, N. "Optimal Execution of Portfolio Transactions", *Journal of Risk*, 2000。
- Bouchaud, J.-P., Bonart, J., Donier, J. & Gould, M. *Trades, Quotes and Prices*（市场冲击与撮合）。
- 第 14 章（WT 原生回放对账）；第 31 章（订单容量与可实现收益）；第 38 章（版本与 manifest）。
